# M1 Task/Activity Label Semantics Revalidation

Reinterpret task/activity window candidates from step 21 as predictive or event-context labels.


## Method
- Primary baseline: `compact13 + random_forest_balanced_depth3 + threshold 0.5`
- Core candidates: 1d, 3d, no-overlap
- Criteria: BA >= 0.75, recall >= 0.8, normal FPR <= 0.25, overlap <= 0.1, group overlap 0


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
OUT = next(p for p in ROOT.iterdir() if p.is_dir() and p.name.startswith('07_'))
THRESHOLD = 0.5
PRIMARY_FEATURE_SET = 'compact13'
PRIMARY_MODEL = 'random_forest_balanced_depth3'
PASS_BA = 0.75
PASS_RECALL = 0.8
PASS_FPR = 0.25
PASS_OVERLAP = 0.10
PASS_FOLD_BA_STD = 0.15
PASS_RECALL_STD = 0.25
PASS_FPR_STD = 0.30

decision = pd.read_csv(OUT / 'm1_task_activity_redesign_decision_matrix.csv')

def window_direction(dataset):
    if '_pre_' in dataset or dataset.endswith('_pre_all'):
        return 'pre'
    if '_post_' in dataset or dataset.endswith('_post_all'):
        return 'post'
    if dataset.endswith('_no_overlap'):
        return 'no_overlap'
    return 'other'

def window_days(dataset):
    if dataset.endswith('_1d'):
        return '1d'
    if dataset.endswith('_3d'):
        return '3d'
    if dataset.endswith('_all'):
        return '7d_reference'
    if dataset.endswith('_no_overlap'):
        return '7d_no_overlap'
    return 'unknown'

sem_metrics = decision.copy()
sem_metrics['window_direction'] = sem_metrics['dataset'].map(window_direction)
sem_metrics['window_days'] = sem_metrics['dataset'].map(window_days)
sem_metrics['core_semantics_candidate'] = sem_metrics['dataset'].map(lambda d: d.endswith('_1d') or d.endswith('_3d') or d.endswith('_no_overlap'))
sem_metrics['semantic_pass'] = (
    sem_metrics['balanced_accuracy'].ge(PASS_BA)
    & sem_metrics['target_recall'].ge(PASS_RECALL)
    & sem_metrics['normal_fpr'].le(PASS_FPR)
    & sem_metrics['target_overlap_rate'].le(PASS_OVERLAP)
    & sem_metrics['max_group_overlap'].eq(0)
    & sem_metrics['fold_balanced_accuracy_std'].le(PASS_FOLD_BA_STD)
    & sem_metrics['fold_target_recall_std'].le(PASS_RECALL_STD)
    & sem_metrics['fold_normal_fpr_std'].le(PASS_FPR_STD)
)
primary = sem_metrics.loc[
    sem_metrics['threshold'].eq(THRESHOLD)
    & sem_metrics['feature_set'].eq(PRIMARY_FEATURE_SET)
    & sem_metrics['model'].eq(PRIMARY_MODEL)
].copy()
primary[['target_class', 'dataset', 'window_direction', 'window_days', 'balanced_accuracy', 'target_recall', 'normal_fpr', 'target_overlap_rate', 'semantic_pass']].sort_values(['target_class', 'window_direction', 'dataset'])


## Generated Artifacts
- `m1_task_activity_semantics_dataset_summary.csv`
- `m1_task_activity_semantics_cv_metrics.csv`
- `m1_task_activity_semantics_decision_matrix.csv`
- `m1_task_activity_semantics_error_audit.csv`
- `m1_task_activity_semantics_feature_importance.csv`
